In [1]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [7]:
import math
import random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="/kaggle/working/data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Device: cuda
Training samples: 60000
Testing samples: 10000


In [8]:
class PredictiveCodingNetwork:

    def __init__(self, dims, activity_lr=0.05, weight_lr=0.001, inference_steps=30):
        self.dims = dims
        self.L = len(dims) - 1
        self.activity_lr = activity_lr
        self.weight_lr = weight_lr
        self.inference_steps = inference_steps

        self.W = [None]

        for l in range(1, len(dims)):
            fan_in = dims[l]
            fan_out = dims[l - 1]
            std = math.sqrt(2.0 / (fan_in + fan_out))
            W = torch.randn(fan_out, fan_in, device=device) * std
            self.W.append(W)

    def phi(self, x, layer):
        if layer == self.L:
            return x
        return torch.tanh(x)

    def phi_prime(self, x, layer):
        if layer == self.L:
            return torch.ones_like(x)

        t = torch.tanh(x)
        return 1.0 - t * t

    def predict_layer(self, x_upper, l):
        return self.phi(x_upper @ self.W[l].T, l - 1)

    def compute_errors(self, x):
        errors = [None] * self.L

        for l in range(1, self.L + 1):
            prediction = self.predict_layer(x[l], l)
            errors[l - 1] = x[l - 1] - prediction

        return errors

    def energy(self, x):
        errors = self.compute_errors(x)
        total = torch.zeros((), device=device)

        for e in errors:
            total += 0.5 * torch.sum(e * e)

        return total

    def initialize_states(self, x_input, y_target=None):
        batch_size = x_input.shape[0]
        x = [None] * (self.L + 1)

        x[0] = x_input.clone()

        for l in range(1, self.L):
            x[l] = torch.zeros(
                batch_size,
                self.dims[l],
                device=device
            )

        if y_target is not None:
            x[self.L] = y_target.clone()
        else:
            x[self.L] = torch.zeros(
                batch_size,
                self.dims[self.L],
                device=device
            )

        return x

    @torch.no_grad()
    def infer(self, x_input, y_target=None, steps=None):

        if steps is None:
            steps = self.inference_steps

        x = self.initialize_states(x_input, y_target)

        for _ in range(steps):

            errors = self.compute_errors(x)

            if y_target is not None:

                for l in range(1, self.L):

                    e_current = errors[l]
                    e_lower = errors[l - 1]

                    feedback = e_lower @ self.W[l]
                    derivative = self.phi_prime(x[l], l)

                    dx = -e_current + derivative * feedback

                    x[l] = x[l] + self.activity_lr * dx

            else:

                l = self.L - 1

                e_current = errors[l]
                e_lower = errors[l - 1]

                feedback = e_lower @ self.W[l]
                derivative = self.phi_prime(x[l], l)

                dx = -e_current + derivative * feedback

                x[l] = x[l] + self.activity_lr * dx

        return x

    @torch.no_grad()
    def update_weights(self, x):

        errors = self.compute_errors(x)

        for l in range(1, self.L + 1):

            e = errors[l - 1]
            pre = self.phi(x[l], l)

            dW = e.T @ pre
            dW /= x[0].shape[0]

            self.W[l] += self.weight_lr * dW

    @torch.no_grad()
    def train_batch(self, x_input, y_target):

        states = self.infer(x_input, y_target)

        energy_before = self.energy(states).item()

        self.update_weights(states)

        return states, energy_before

In [9]:
model = PredictiveCodingNetwork(
    dims=[784, 128, 10],
    activity_lr=0.05,
    weight_lr=0.001,
    inference_steps=30
)

print("Predictive Coding Network")
print("784 -> 128 -> 10")

Predictive Coding Network
784 -> 128 -> 10


In [10]:
def one_hot(labels, num_classes=10):
    return F.one_hot(
        labels,
        num_classes=num_classes
    ).float()

In [11]:
EPOCHS = 5

for epoch in range(EPOCHS):

    total_energy = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.view(images.shape[0], -1).to(device)
        labels = labels.to(device)

        targets = one_hot(
            labels,
            num_classes=10
        ).to(device)

        states, energy = model.train_batch(
            images,
            targets
        )

        total_energy += energy

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}] "
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"Energy: {energy:.4f}"
            )

    avg_energy = total_energy / len(train_loader)

    print(f"Epoch {epoch+1} completed")
    print(f"Average Energy: {avg_energy:.4f}")
    print("-" * 60)

Epoch [1/5] Batch [100/469] Energy: 15559.5869
Epoch [1/5] Batch [200/469] Energy: 13504.4756
Epoch [1/5] Batch [300/469] Energy: 12152.5664
Epoch [1/5] Batch [400/469] Energy: 11391.2158
Epoch 1 completed
Average Energy: 14209.9863
------------------------------------------------------------
Epoch [2/5] Batch [100/469] Energy: 10516.7598
Epoch [2/5] Batch [200/469] Energy: 9357.5908
Epoch [2/5] Batch [300/469] Energy: 9037.1611
Epoch [2/5] Batch [400/469] Energy: 8756.6699
Epoch 2 completed
Average Energy: 9636.0584
------------------------------------------------------------
Epoch [3/5] Batch [100/469] Energy: 8205.9512
Epoch [3/5] Batch [200/469] Energy: 8145.3936
Epoch [3/5] Batch [300/469] Energy: 7610.5674
Epoch [3/5] Batch [400/469] Energy: 7432.1523
Epoch 3 completed
Average Energy: 7941.6059
------------------------------------------------------------
Epoch [4/5] Batch [100/469] Energy: 7289.6216
Epoch [4/5] Batch [200/469] Energy: 6896.3940
Epoch [4/5] Batch [300/469] Energy:

In [12]:
@torch.no_grad()
def evaluate(model, test_loader):

    correct = 0
    total = 0

    for images, labels in test_loader:

        images = images.view(images.shape[0], -1).to(device)
        labels = labels.to(device)

        states = model.infer(
            images,
            y_target=None
        )

        output = states[-1]

        predictions = output.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = 100.0 * correct / total

    return accuracy

In [13]:
accuracy = evaluate(
    model,
    test_loader
)

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 9.80%
